# Feature Engineering + Principal Component Analysis

This notebook:
1. **Engineers many features** from existing Landsat bands (indices, ratios, interactions)
2. **Extracts ALL 14 TerraClimate variables** (requires Planetary Computer API)
3. **Runs PCA** on the full feature set to understand variance structure

---

## 1. Load Dependencies

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from IPython.display import display

pd.set_option('display.max_columns', 50)
print('Dependencies loaded.')

## 2. Load Existing Data

In [ ]:
# Load all existing CSVs
wq_train = pd.read_csv('water_quality_training_dataset.csv')
landsat_train = pd.read_csv('landsat_features_training.csv')
terra_train = pd.read_csv('terraclimate_features_training.csv')

landsat_val = pd.read_csv('landsat_features_validation.csv')
terra_val = pd.read_csv('terraclimate_features_validation.csv')
submission = pd.read_csv('submission_template.csv')

print('Training shapes:')
print(f'  Water quality: {wq_train.shape}')
print(f'  Landsat:       {landsat_train.shape}')
print(f'  TerraClimate:  {terra_train.shape}')
print(f'\nValidation shapes:')
print(f'  Landsat:       {landsat_val.shape}')
print(f'  TerraClimate:  {terra_val.shape}')
print(f'  Submission:    {submission.shape}')

## 3. Landsat Feature Engineering

From the 4 existing bands (`nir`, `green`, `swir16`, `swir22`) we can derive many additional features:
- **Spectral indices** (NDWI, moisture variants)
- **Band ratios** (all pairwise)
- **Band statistics** (mean, std, range across bands)
- **Interaction terms**

In [ ]:
def engineer_landsat_features(df):
    """Engineer all possible features from existing Landsat bands."""
    out = df.copy()
    eps = 1e-10
    
    nir = out['nir'].astype(float)
    green = out['green'].astype(float)
    swir16 = out['swir16'].astype(float)
    swir22 = out['swir22'].astype(float)
    
    # ---- Normalized Difference Indices ----
    # NDWI (Normalized Difference Water Index) - water body detection
    out['NDWI'] = (green - nir) / (green + nir + eps)
    
    # NDMI already exists, but recompute to be safe
    out['NDMI'] = (nir - swir16) / (nir + swir16 + eps)
    
    # MNDWI already exists
    out['MNDWI'] = (green - swir16) / (green + swir16 + eps)
    
    # Moisture Index variant 2: NIR vs SWIR22
    out['MI2'] = (nir - swir22) / (nir + swir22 + eps)
    
    # MNDWI variant using SWIR22 instead of SWIR16
    out['MNDWI2'] = (green - swir22) / (green + swir22 + eps)
    
    # SWIR difference index
    out['SWIR_NDI'] = (swir16 - swir22) / (swir16 + swir22 + eps)
    
    # ---- Band Ratios (all pairwise, 6 combinations) ----
    out['nir_green_ratio'] = nir / (green + eps)
    out['nir_swir16_ratio'] = nir / (swir16 + eps)
    out['nir_swir22_ratio'] = nir / (swir22 + eps)
    out['green_swir16_ratio'] = green / (swir16 + eps)
    out['green_swir22_ratio'] = green / (swir22 + eps)
    out['swir16_swir22_ratio'] = swir16 / (swir22 + eps)
    
    # ---- Band Statistics (per-sample across the 4 bands) ----
    band_stack = np.column_stack([nir, green, swir16, swir22])
    out['band_mean'] = np.nanmean(band_stack, axis=1)
    out['band_std'] = np.nanstd(band_stack, axis=1)
    out['band_range'] = np.nanmax(band_stack, axis=1) - np.nanmin(band_stack, axis=1)
    out['band_max'] = np.nanmax(band_stack, axis=1)
    out['band_min'] = np.nanmin(band_stack, axis=1)
    out['band_cv'] = out['band_std'] / (out['band_mean'] + eps)  # coefficient of variation
    
    # ---- Log-transformed bands (useful for skewed reflectance) ----
    out['log_nir'] = np.log1p(nir)
    out['log_green'] = np.log1p(green)
    out['log_swir16'] = np.log1p(swir16)
    out['log_swir22'] = np.log1p(swir22)
    
    # ---- Interaction terms ----
    out['nir_x_green'] = nir * green
    out['swir16_x_swir22'] = swir16 * swir22
    out['nir_x_swir16'] = nir * swir16
    out['green_x_swir22'] = green * swir22
    
    return out

print('Feature engineering function defined.')

In [ ]:
# Apply to training and validation
landsat_train_fe = engineer_landsat_features(landsat_train)
landsat_val_fe = engineer_landsat_features(landsat_val)

# Show new columns
new_cols = [c for c in landsat_train_fe.columns if c not in landsat_train.columns]
print(f'Original Landsat columns: {len(landsat_train.columns)}')
print(f'After engineering: {len(landsat_train_fe.columns)}')
print(f'New features ({len(new_cols)}): {new_cols}')
display(landsat_train_fe.head())

## 4. Extract ALL TerraClimate Variables

The existing data only has `pet`. TerraClimate has **14 variables** total:

| Variable | Description |
|----------|-------------|
| `aet` | Actual Evapotranspiration |
| `def` | Climate Water Deficit |
| `pet` | Potential Evapotranspiration (already have) |
| `ppt` | Precipitation |
| `q` | Runoff |
| `soil` | Soil Moisture |
| `srad` | Downward Shortwave Radiation |
| `swe` | Snow Water Equivalent |
| `tmax` | Max Temperature |
| `tmin` | Min Temperature |
| `vap` | Vapor Pressure |
| `vpd` | Vapor Pressure Deficit |
| `ws` | Wind Speed |
| `PDSI` | Palmer Drought Severity Index |

**The cell below extracts all 14 variables.** It requires Planetary Computer API access (works on Snowflake or locally with internet). If running locally without API access, skip to the fallback cell that works with just `pet`.

In [ ]:
# ============================================================
# TerraClimate Extraction - ALL 14 variables
# Requires: pystac_client, planetary_computer, xarray, scipy
# ============================================================

EXTRACT_TERRACLIMATE = False  # <-- Set to True to run extraction

if EXTRACT_TERRACLIMATE:
    import xarray as xr
    from scipy.spatial import cKDTree
    import pystac_client
    import planetary_computer as pc
    from tqdm import tqdm

    def load_terraclimate_dataset():
        catalog = pystac_client.Client.open(
            "https://planetarycomputer.microsoft.com/api/stac/v1",
            modifier=pc.sign_inplace,
        )
        collection = catalog.get_collection("terraclimate")
        asset = collection.assets["zarr-abfs"]
        if "xarray:storage_options" in asset.extra_fields:
            ds = xr.open_zarr(
                asset.href,
                storage_options=asset.extra_fields["xarray:storage_options"],
                consolidated=True,
            )
        else:
            ds = xr.open_dataset(asset.href, **asset.extra_fields["xarray:open_kwargs"])
        return ds

    def filterg(ds, var):
        ds_filtered = ds[var].sel(time=slice("2011-01-01", "2015-12-31"))
        df_list = []
        for i in tqdm(range(len(ds_filtered.time)), desc=f'Filtering {var}'):
            df_var = ds_filtered.isel(time=i).to_dataframe().reset_index()
            df_var = df_var[
                (df_var['lat'] > -35.18) & (df_var['lat'] < -21.72) &
                (df_var['lon'] > 14.97) & (df_var['lon'] < 32.79)
            ]
            df_list.append(df_var)
        df_final = pd.concat(df_list, ignore_index=True)
        df_final['time'] = df_final['time'].astype(str)
        df_final = df_final.rename(columns={'lat': 'Latitude', 'lon': 'Longitude', 'time': 'Sample Date'})
        return df_final

    def assign_nearest_climate(sa_df, climate_df, var_name):
        sa_coords = np.radians(sa_df[['Latitude', 'Longitude']].values)
        climate_coords = np.radians(climate_df[['Latitude', 'Longitude']].values)
        tree = cKDTree(climate_coords)
        dist, idx = tree.query(sa_coords, k=1)
        nearest_points = climate_df.iloc[idx].reset_index(drop=True)
        sa_df = sa_df.reset_index(drop=True)
        sa_df[['nearest_lat', 'nearest_lon']] = nearest_points[['Latitude', 'Longitude']]
        sa_df['Sample Date'] = pd.to_datetime(sa_df['Sample Date'], dayfirst=True, errors='coerce')
        climate_df['Sample Date'] = pd.to_datetime(climate_df['Sample Date'], dayfirst=True, errors='coerce')
        values = []
        for i in tqdm(range(len(sa_df)), desc=f'Mapping {var_name}'):
            sample_date = sa_df.loc[i, 'Sample Date']
            subset = climate_df[
                (climate_df['Latitude'] == sa_df.loc[i, 'nearest_lat']) &
                (climate_df['Longitude'] == sa_df.loc[i, 'nearest_lon'])
            ]
            if subset.empty:
                values.append(np.nan)
                continue
            nearest_idx = (subset['Sample Date'] - sample_date).abs().idxmin()
            values.append(subset.loc[nearest_idx, var_name])
        return pd.DataFrame({var_name: values})

    # --- Extract all variables ---
    tc_vars = ['aet', 'def', 'pet', 'ppt', 'q', 'soil', 'srad',
               'swe', 'tmax', 'tmin', 'vap', 'vpd', 'ws', 'PDSI']

    ds = load_terraclimate_dataset()

    # Training set
    terra_train_all = wq_train[['Latitude', 'Longitude', 'Sample Date']].copy()
    for var in tc_vars:
        print(f'\n--- Extracting {var} for training ---')
        tc_filtered = filterg(ds, var)
        var_df = assign_nearest_climate(wq_train, tc_filtered, var)
        terra_train_all[var] = var_df[var].values

    terra_train_all.to_csv('terraclimate_all_features_training.csv', index=False)
    print(f'\nSaved terraclimate_all_features_training.csv with shape {terra_train_all.shape}')

    # Validation set
    terra_val_all = submission[['Latitude', 'Longitude', 'Sample Date']].copy()
    for var in tc_vars:
        print(f'\n--- Extracting {var} for validation ---')
        tc_filtered = filterg(ds, var)
        var_df = assign_nearest_climate(submission, tc_filtered, var)
        terra_val_all[var] = var_df[var].values

    terra_val_all.to_csv('terraclimate_all_features_validation.csv', index=False)
    print(f'Saved terraclimate_all_features_validation.csv with shape {terra_val_all.shape}')
    
    terra_train = terra_train_all
    terra_val = terra_val_all

else:
    print('Skipping TerraClimate extraction. Using existing pet-only data.')
    print('Set EXTRACT_TERRACLIMATE = True to extract all 14 variables.')

### 4b. TerraClimate Feature Engineering

If you have all 14 variables, we can derive additional features. If only `pet` is available, we still engineer what we can.

In [ ]:
def engineer_terraclimate_features(df):
    """Engineer features from TerraClimate variables."""
    out = df.copy()
    eps = 1e-10
    
    # If we have the full set of variables, derive interactions
    if 'tmax' in out.columns and 'tmin' in out.columns:
        out['temp_range'] = out['tmax'] - out['tmin']
        out['temp_mean'] = (out['tmax'] + out['tmin']) / 2
    
    if 'ppt' in out.columns and 'pet' in out.columns:
        out['aridity_index'] = out['ppt'] / (out['pet'] + eps)  # P/PET
        out['moisture_surplus'] = out['ppt'] - out['pet']
    
    if 'aet' in out.columns and 'pet' in out.columns:
        out['evap_fraction'] = out['aet'] / (out['pet'] + eps)  # AET/PET
        out['water_stress'] = out['pet'] - out['aet']
    
    if 'ppt' in out.columns and 'q' in out.columns:
        out['runoff_ratio'] = out['q'] / (out['ppt'] + eps)
    
    if 'vap' in out.columns and 'vpd' in out.columns:
        out['vap_total'] = out['vap'] + out['vpd']  # saturated vapor pressure
        out['relative_humidity_proxy'] = out['vap'] / (out['vap'] + out['vpd'] + eps)
    
    if 'srad' in out.columns and 'tmax' in out.columns:
        out['radiation_temp_interaction'] = out['srad'] * out['tmax']
    
    if 'ws' in out.columns and 'pet' in out.columns:
        out['wind_evap_interaction'] = out['ws'] * out['pet']
    
    if 'soil' in out.columns and 'ppt' in out.columns:
        out['soil_ppt_ratio'] = out['soil'] / (out['ppt'] + eps)
    
    return out

terra_train_fe = engineer_terraclimate_features(terra_train)
terra_val_fe = engineer_terraclimate_features(terra_val)

terra_new = [c for c in terra_train_fe.columns if c not in terra_train.columns]
print(f'TerraClimate: {len(terra_train.columns)} -> {len(terra_train_fe.columns)} columns')
print(f'New features ({len(terra_new)}): {terra_new}')

## 5. Temporal & Spatial Features

Extract useful information from the sample date and coordinates.

In [ ]:
def add_temporal_spatial_features(df, date_col='Sample Date'):
    """Add temporal and spatial features."""
    out = df.copy()
    dt = pd.to_datetime(out[date_col], dayfirst=True, errors='coerce')
    
    out['year'] = dt.dt.year
    out['month'] = dt.dt.month
    out['day_of_year'] = dt.dt.dayofyear
    
    # Cyclical encoding for month (captures seasonality)
    out['month_sin'] = np.sin(2 * np.pi * out['month'] / 12)
    out['month_cos'] = np.cos(2 * np.pi * out['month'] / 12)
    
    # Cyclical encoding for day of year
    out['doy_sin'] = np.sin(2 * np.pi * out['day_of_year'] / 365)
    out['doy_cos'] = np.cos(2 * np.pi * out['day_of_year'] / 365)
    
    # Season (Southern Hemisphere)
    # Summer: Dec-Feb, Autumn: Mar-May, Winter: Jun-Aug, Spring: Sep-Nov
    season_map = {12: 0, 1: 0, 2: 0,   # summer
                  3: 1, 4: 1, 5: 1,      # autumn
                  6: 2, 7: 2, 8: 2,      # winter
                  9: 3, 10: 3, 11: 3}    # spring
    out['season'] = out['month'].map(season_map)
    
    return out

# Apply to water quality data (which has the dates)
wq_train_fe = add_temporal_spatial_features(wq_train)
submission_fe = add_temporal_spatial_features(submission)

temporal_cols = ['year', 'month', 'day_of_year', 'month_sin', 'month_cos', 'doy_sin', 'doy_cos', 'season']
print(f'Temporal features added: {temporal_cols}')

## 6. Combine All Features

In [ ]:
# --- Identify feature columns from each source ---
id_cols = ['Latitude', 'Longitude', 'Sample Date']
target_cols = ['Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus']

landsat_feature_cols = [c for c in landsat_train_fe.columns if c not in id_cols]
terra_feature_cols = [c for c in terra_train_fe.columns if c not in id_cols]

# --- Build combined training dataframe ---
train_combined = wq_train_fe.copy()

# Add Landsat features
for col in landsat_feature_cols:
    train_combined[col] = landsat_train_fe[col].values

# Add TerraClimate features
for col in terra_feature_cols:
    train_combined[col] = terra_train_fe[col].values

# --- Build combined validation dataframe ---
val_combined = submission_fe.copy()

for col in landsat_feature_cols:
    val_combined[col] = landsat_val_fe[col].values

for col in terra_feature_cols:
    val_combined[col] = terra_val_fe[col].values

# --- Identify all numeric feature columns ---
all_feature_cols = [c for c in train_combined.columns 
                    if c not in id_cols + target_cols + ['Sample Date']
                    and pd.api.types.is_numeric_dtype(train_combined[c])]

print(f'Total features for PCA: {len(all_feature_cols)}')
print(f'\nAll feature columns:')
for i, c in enumerate(all_feature_cols, 1):
    print(f'  {i:2d}. {c}')

In [ ]:
# Handle missing values and infinities
train_features = train_combined[all_feature_cols].copy()
train_features = train_features.replace([np.inf, -np.inf], np.nan)
train_features = train_features.fillna(train_features.median())

val_features = val_combined[all_feature_cols].copy()
val_features = val_features.replace([np.inf, -np.inf], np.nan)
val_features = val_features.fillna(val_features.median())

print(f'Training features shape: {train_features.shape}')
print(f'Validation features shape: {val_features.shape}')
print(f'Any NaN remaining (train): {train_features.isna().any().any()}')
print(f'Any NaN remaining (val):   {val_features.isna().any().any()}')

## 7. Principal Component Analysis

PCA finds orthogonal directions of maximum variance. We:
1. Standardize all features (mean=0, std=1)
2. Fit PCA on training data
3. Analyze explained variance
4. Visualize components and loadings

In [ ]:
# Standardize
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(train_features)
X_val_scaled = scaler.transform(val_features)

# Fit PCA with all components
pca_full = PCA()
X_train_pca = pca_full.fit_transform(X_train_scaled)
X_val_pca = pca_full.transform(X_val_scaled)

print(f'PCA fitted on {X_train_scaled.shape[1]} features')
print(f'Number of components: {pca_full.n_components_}')

### 7a. Explained Variance

In [ ]:
# Explained variance table
var_explained = pd.DataFrame({
    'Component': [f'PC{i+1}' for i in range(len(pca_full.explained_variance_ratio_))],
    'Explained Variance Ratio': pca_full.explained_variance_ratio_,
    'Cumulative Variance': np.cumsum(pca_full.explained_variance_ratio_)
})

print('Explained Variance by Component:')
display(var_explained.head(20))

# How many components for 90%, 95%, 99%?
for threshold in [0.80, 0.90, 0.95, 0.99]:
    n = np.argmax(np.cumsum(pca_full.explained_variance_ratio_) >= threshold) + 1
    print(f'{threshold*100:.0f}% variance explained by {n} components (out of {len(pca_full.explained_variance_ratio_)})')

In [ ]:
# Scree plot + cumulative variance
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

n_show = min(30, len(pca_full.explained_variance_ratio_))

# Scree plot
ax1.bar(range(1, n_show+1), pca_full.explained_variance_ratio_[:n_show], alpha=0.7, color='steelblue')
ax1.set_xlabel('Principal Component')
ax1.set_ylabel('Explained Variance Ratio')
ax1.set_title('Scree Plot')
ax1.set_xticks(range(1, n_show+1))

# Cumulative variance
cumvar = np.cumsum(pca_full.explained_variance_ratio_)
ax2.plot(range(1, len(cumvar)+1), cumvar, 'o-', color='steelblue', markersize=4)
ax2.axhline(y=0.90, color='red', linestyle='--', label='90% threshold')
ax2.axhline(y=0.95, color='orange', linestyle='--', label='95% threshold')
ax2.set_xlabel('Number of Components')
ax2.set_ylabel('Cumulative Explained Variance')
ax2.set_title('Cumulative Explained Variance')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 7b. PCA Loadings (Feature Contributions)

Loadings show how much each original feature contributes to each principal component.

In [ ]:
# Loadings for first 5 components
n_components_show = min(5, pca_full.n_components_)
loadings = pd.DataFrame(
    pca_full.components_[:n_components_show].T,
    index=all_feature_cols,
    columns=[f'PC{i+1}' for i in range(n_components_show)]
)

print('Top feature loadings per component:\n')
for pc in loadings.columns:
    top = loadings[pc].abs().sort_values(ascending=False).head(8)
    print(f'--- {pc} (explains {pca_full.explained_variance_ratio_[int(pc[2:])-1]*100:.1f}% variance) ---')
    for feat, val in top.items():
        sign = '+' if loadings.loc[feat, pc] > 0 else '-'
        print(f'  {sign} {feat}: {loadings.loc[feat, pc]:.4f}')
    print()

In [ ]:
# Heatmap of loadings (top features)
n_top = min(20, len(all_feature_cols))

# Select features with highest total absolute loading across first 5 PCs
total_importance = loadings.abs().sum(axis=1).sort_values(ascending=False)
top_features = total_importance.head(n_top).index.tolist()

fig, ax = plt.subplots(figsize=(10, max(8, n_top * 0.4)))
sns.heatmap(
    loadings.loc[top_features],
    annot=True, fmt='.2f', cmap='RdBu_r', center=0,
    ax=ax, linewidths=0.5
)
ax.set_title(f'PCA Loadings - Top {n_top} Features by Importance')
ax.set_xlabel('Principal Component')
plt.tight_layout()
plt.show()

### 7c. PCA Scatter Plots

Visualize training samples in PC space, colored by each target variable.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, target in zip(axes, target_cols):
    y = train_combined[target].values
    # Clip to 1st-99th percentile for better color scaling
    vmin, vmax = np.nanpercentile(y, [1, 99])
    sc = ax.scatter(
        X_train_pca[:, 0], X_train_pca[:, 1],
        c=y, cmap='viridis', alpha=0.4, s=5,
        vmin=vmin, vmax=vmax
    )
    ax.set_xlabel(f'PC1 ({pca_full.explained_variance_ratio_[0]*100:.1f}%)')
    ax.set_ylabel(f'PC2 ({pca_full.explained_variance_ratio_[1]*100:.1f}%)')
    ax.set_title(target)
    plt.colorbar(sc, ax=ax)

plt.suptitle('Training Samples in PC1-PC2 Space', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Train vs Validation distribution in PC space
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(X_train_pca[:, 0], X_train_pca[:, 1], c='steelblue', alpha=0.3, s=5, label='Training')
ax.scatter(X_val_pca[:, 0], X_val_pca[:, 1], c='red', alpha=0.7, s=20, label='Validation')
ax.set_xlabel(f'PC1 ({pca_full.explained_variance_ratio_[0]*100:.1f}%)')
ax.set_ylabel(f'PC2 ({pca_full.explained_variance_ratio_[1]*100:.1f}%)')
ax.set_title('Training vs Validation in PC Space')
ax.legend()
plt.tight_layout()
plt.show()

### 7d. PC Correlation with Targets

Which principal components correlate most strongly with each water quality target?

In [ ]:
# Correlate PCs with targets
n_pcs_check = min(15, pca_full.n_components_)
pc_df = pd.DataFrame(X_train_pca[:, :n_pcs_check], columns=[f'PC{i+1}' for i in range(n_pcs_check)])

for target in target_cols:
    pc_df[target] = train_combined[target].values

corr_matrix = pc_df.corr()

# Extract just PC-target correlations
pc_names = [f'PC{i+1}' for i in range(n_pcs_check)]
target_corr = corr_matrix.loc[pc_names, target_cols]

fig, ax = plt.subplots(figsize=(8, max(5, n_pcs_check * 0.35)))
sns.heatmap(target_corr, annot=True, fmt='.3f', cmap='RdBu_r', center=0,
            ax=ax, linewidths=0.5)
ax.set_title('Correlation: Principal Components vs Water Quality Targets')
plt.tight_layout()
plt.show()

print('\nStrongest PC-Target correlations:')
for target in target_cols:
    best_pc = target_corr[target].abs().idxmax()
    best_val = target_corr.loc[best_pc, target]
    print(f'  {target}: {best_pc} (r = {best_val:.3f})')

## 8. Export PCA-Transformed Features

Save PCA-transformed features for use in modeling. Choose the number of components that captures enough variance (e.g., 95%).

In [ ]:
# Choose components for 95% variance
target_variance = 0.95
n_components_95 = np.argmax(np.cumsum(pca_full.explained_variance_ratio_) >= target_variance) + 1
print(f'Using {n_components_95} components for {target_variance*100:.0f}% variance')

# Refit with chosen number
pca_reduced = PCA(n_components=n_components_95)
X_train_reduced = pca_reduced.fit_transform(X_train_scaled)
X_val_reduced = pca_reduced.transform(X_val_scaled)

# Create DataFrames
pc_cols = [f'PC{i+1}' for i in range(n_components_95)]

train_pca_df = pd.DataFrame(X_train_reduced, columns=pc_cols)
train_pca_df['Latitude'] = wq_train['Latitude'].values
train_pca_df['Longitude'] = wq_train['Longitude'].values
train_pca_df['Sample Date'] = wq_train['Sample Date'].values
train_pca_df['Total Alkalinity'] = wq_train['Total Alkalinity'].values
train_pca_df['Electrical Conductance'] = wq_train['Electrical Conductance'].values
train_pca_df['Dissolved Reactive Phosphorus'] = wq_train['Dissolved Reactive Phosphorus'].values

val_pca_df = pd.DataFrame(X_val_reduced, columns=pc_cols)
val_pca_df['Latitude'] = submission['Latitude'].values
val_pca_df['Longitude'] = submission['Longitude'].values
val_pca_df['Sample Date'] = submission['Sample Date'].values

# Save
train_pca_df.to_csv('pca_features_training.csv', index=False)
val_pca_df.to_csv('pca_features_validation.csv', index=False)

print(f'\nSaved pca_features_training.csv: {train_pca_df.shape}')
print(f'Saved pca_features_validation.csv: {val_pca_df.shape}')
display(train_pca_df.head())

## 9. Feature Summary

In [ ]:
print('=' * 60)
print('FEATURE ENGINEERING SUMMARY')
print('=' * 60)
print(f'\nOriginal features:')
print(f'  Landsat bands:    4 (nir, green, swir16, swir22)')
print(f'  Landsat indices:  2 (NDMI, MNDWI)')
print(f'  TerraClimate:     {len([c for c in terra_train.columns if c not in id_cols])} ({", ".join([c for c in terra_train.columns if c not in id_cols])})')
print(f'\nEngineered features:')
print(f'  Total features:   {len(all_feature_cols)}')
print(f'\nPCA results:')
print(f'  Components for 95% variance: {n_components_95}')
print(f'  Dimensionality reduction: {len(all_feature_cols)} -> {n_components_95}')
print(f'\nFiles saved:')
print(f'  pca_features_training.csv')
print(f'  pca_features_validation.csv')